# Installing important libraries

In [ ]:
!pip install  openai langchain faiss-cpu pypdf tiktoken docarray PyPDF tiktoken langchain-openai flashrank langchain-community pillow


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
from dotenv import load_dotenv
import os
import shutil
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


In [41]:
load_dotenv()

True

In [43]:
# Load PDF documents with error handling
print("Loading PDF documents from ./Policy+Documents...")
try:
    pdf_directory_loader = PyPDFDirectoryLoader("./Policy+Documents")
    documents = pdf_directory_loader.load()
    print(f"✓ Successfully loaded {len(documents)} documents")
    print(f"✓ Total pages: {sum(doc.metadata.get('total_pages', 1) for doc in documents)}")
except Exception as e:
    print(f"Error loading documents: {str(e)}")
    raise

Loading PDF documents from ./Policy+Documents...
✓ Successfully loaded 217 documents
✓ Total pages: 7209
✓ Successfully loaded 217 documents
✓ Total pages: 7209


In [44]:
documents[0].page_content[:100]

'Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s Contact Num'

In [45]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 760 document chunks
✓ Average chunk size: 848 characters


In [46]:
print(splits[0])

page_content='Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Your Policy no. <<  >> 
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health (“Policy”) 
being this document, has been issued. We have made every effort to design your Policy in a simple format. We 
have highlighted items of importance so that you may recognize them easily. 
 
Policy document: 
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy 
is enclosed herewith. Please preserve this document safely and also inform your nominees about the same. A 
copy of your proposal form and other relevant documents submitted by you is also enclosed for your 
information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the Policy, you have the option to' metad

In [47]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized


In [48]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✓ Test embedding successful - dimension: 1536


In [49]:
# store = LocalFileStore("./cache/") 

# cached_embedder = CacheBackedEmbeddings.from_bytes_store(
#     embeddings_model,
#     store,
#     namespace="semantic-spotter"
# )

In [50]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Yo...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: option to return the Policy to us for cancellation stating the reasons thereof, within 15 days from the date of 
receipt of the Policy. On receipt of ...

✓ Total splits available: 760


In [51]:

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """Create and save FAISS vector store."""
    print(f"Creating vector store from {len(splits)} documents...")
    start_time = time.time()
    
    try:
        # Create FAISS store directly from documents
        vectordb = FAISS.from_documents(
            documents=splits,
            embedding=embeddings_model
        )
        print(f"✓ FAISS vector store created")
        
        # Save to disk
        os.makedirs(save_path, exist_ok=True)
        vectordb.save_local(save_path)
        print(f"✓ Saved to: {save_path}")
        
        elapsed = time.time() - start_time
        print(f"✓ Time: {elapsed:.1f}s ({elapsed/60:.1f}m)")
        
        return vectordb
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {e}")
        raise

# Create the vector store
try:
    vectordb = create_vector_store_faiss(splits, embeddings_model, "./faiss_store")
    print("✓ Vector store ready for similarity search")
except Exception as e:
    print(f"Failed to create vector store: {str(e)}")
    raise


Creating vector store from 760 documents...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✓ FAISS vector store created
✓ Saved to: ./faiss_store
✓ Time: 6.6s (0.1m)
✓ Vector store ready for similarity search


In [52]:
# Function to load existing vector store (FAISS only)
def load_vector_store(store_path="./faiss_store", store_type="faiss"):
    """
    Load an existing vector store from disk.
    Only FAISS stores are supported in this notebook.
    """
    try:
        if store_type.lower() == "faiss":
            if not os.path.exists(store_path):
                print(f"✗ FAISS vector store not found at: {store_path}")
                return None

            vectordb = FAISS.load_local(
                folder_path=store_path,
                embeddings=embeddings_model,
                allow_dangerous_deserialization=True,
            )
            print(f"✓ Successfully loaded FAISS vector store from: {store_path}")
            return vectordb
        else:
            print(f"✗ Unsupported store_type: {store_type}. Only 'faiss' is supported in this notebook.")
            return None
    except Exception as e:
        print(f"✗ Error loading vector store: {str(e)}")
        return None

# Uncomment to load existing vector store:
# vectordb = load_vector_store("./faiss_store", "faiss")


In [53]:
# Install required packages for advanced retrieval
%pip install -q langchain-community sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
vectordb = load_vector_store()

✓ Successfully loaded FAISS vector store from: ./faiss_store


In [55]:
vectordb.as_retriever()

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000287D720EC50>, search_kwargs={})

In [56]:
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
)

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""
    retrieved_docs = compression_retriever.invoke(
    query
)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [57]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

# Instantiate the LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", streaming=True)

tools = [retrieve_context]

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]
2. Do NOT include any explanation, preamble, or extra text
3. Do NOT repeat the question
4. Do NOT include metadata (producer, creator, page, author, etc.)
5. Use only the actual policy content
6. If answer not found, write: "Not found in the provided policy context."

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

agent = create_agent(llm, tools, system_prompt=prompt)


In [58]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    response = agent.invoke({
        'messages': [
            HumanMessage(content=(
            query
            ))
        ]
    })

    print(response['messages'][-1].content)

In [59]:
insurance_agent( "What is the life insurance policy coverage amount?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The life insurance coverage amount varies as it is determined by the Sum Assured chosen by the Scheme Member, which may differ among members. However, the maximum benefit payable for Accidental Death under all policies combined is limited to Rs. 10,000,000 (Rupees 1 crore only).
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, page 6 and Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions, page 10.


In [60]:
insurance_agent( "Can a 100 year plus person do a term insurance?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The eligibility for term insurance generally includes age limits, with a maximum age set by the insurer. A person over 100 years of age may not meet the maximum entry age criteria for term insurance as specified in the policy documents. 

source: HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf, page 15


In [61]:
insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The definitions of covered Critical Illnesses include conditions such as Myocardial Infarction, which is defined as the first occurrence of a heart attack characterized by specific clinical symptoms, ECG changes, and elevation of specific enzymes. Exclusions apply to other acute coronary syndromes and certain conditions without overt ischemic heart disease. 
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, page 27


In [62]:
# retrieve_context("what is the Definitions of Critical Illnesses? based on policy?")

In [63]:
# retrieve_context("Can a 100 year plus person do a term insurance?")

In [64]:
insurance_agent("what is the life insurance coverage for disability?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The policy does not specify coverage for disability under life insurance, focusing instead on critical illnesses and death benefits, with no mention of specific disability coverage. 
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, page 7.
